# SOOP Korean to Chinese SRT and hard-sub videos

This notebook uses Google Drive for model cache, outputs, and resume state. First pass creates `ko.srt`; download it and translate it locally with `translate-srt`; upload/save `zh.srt`; second pass uses a CSV file to render high-resolution hard-sub clips.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If this notebook is not already inside the project directory, set REPO_URL and run this cell.
REPO_URL = 'https://github.com/digiprospector/kr-sc-srt.git'  # Example: 'https://github.com/your-name/kr-sc-srt.git'
PROJECT_DIR = '/content/kr-sc-srt'

import os, pathlib, subprocess
project_path = pathlib.Path(PROJECT_DIR)
if REPO_URL and not project_path.exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
elif REPO_URL and (project_path / '.git').exists():
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], check=True)
if project_path.exists():
    %cd {PROJECT_DIR}
else:
    print('Set REPO_URL or upload/open this notebook from the project directory.')

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q -U pip
!python -m pip install -q -r requirements.txt yt-dlp

In [ ]:
import os
import subprocess
import sys

ROOT = '/content/drive/MyDrive/kr-sc-srt'
MODEL_CACHE = f'{ROOT}/models'
LAST_JOB = f'{ROOT}/last_job.json'

# First run: paste the VOD URL. Later reruns can leave VOD_URL empty and use RESUME_LAST.
VOD_URL = ''
RESUME_LAST = True

os.makedirs(ROOT, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

if not VOD_URL and not os.path.exists(LAST_JOB):
    VOD_URL = input('First run VOD URL: ').strip()
    if not VOD_URL:
        raise ValueError('VOD_URL is required for the first run.')

def run_cli(cmd):
    print(' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)

In [ ]:
cmd = [sys.executable, '-u', '-m', 'kr_sc_srt', 'prepare', '--root', ROOT, '--model-cache-dir', MODEL_CACHE]
if VOD_URL:
    cmd.append(VOD_URL)
elif RESUME_LAST:
    if not os.path.exists(LAST_JOB):
        raise ValueError(f'No saved job found at {LAST_JOB}. Set VOD_URL for the first run.')
    cmd.append('--resume-last')
else:
    raise ValueError('Set VOD_URL or enable RESUME_LAST')

run_cli(cmd)

raise SystemExit('\n=== PREPARE COMPLETED ===\n\n1. Download ko.srt and translate it to zh.srt\n2. Upload zh.srt and your segment CSV back to Google Drive\n3. Manually run the Render cell below.')

After prepare completes, download `ko.srt` and translate it on your local machine:

```bash
python -m kr_sc_srt translate-srt ko.srt zh.srt --api-key-env OPENAI_API_KEY
```

Upload/save `zh.srt` back into the same job output directory. Then create or edit the segment CSV in that directory. Format:

```csv
name,start,end
part-01,00:01:30,00:05:00
part-02,00:05:00,00:08:30.500
```

By default, `render --resume-last` looks for `<job-name>.csv` inside the saved output directory.

In [ ]:
cmd = [sys.executable, '-u', '-m', 'kr_sc_srt', 'render', '--root', ROOT, '--resume-last']
if not os.path.exists(LAST_JOB):
    raise ValueError(f'No saved job found at {LAST_JOB}. Run prepare once with VOD_URL first.')
run_cli(cmd)

if os.path.exists(LAST_JOB):
    os.remove(LAST_JOB)
    print('Completed successfully. Deleted last_job.json to clean up state.')